# Rank-three model with a close spectrum

This experiment evaluates joint FSNM estimation when the three true singular values are close. Individual singular functions are then weakly identifiable, so subspace recovery is the relevant metric.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import ParameterGrid

from fsnm import empirical_loss, fit_fsnm


def basis_matrix(values, rank=3):
    values = np.asarray(values)
    frequencies = np.arange(1, rank + 1)
    return np.sqrt(2) * np.cos(np.pi * values[:, None] * frequencies)


def kappa_exact(x_values, y_values):
    return 1 + (basis_matrix(x_values) * SIGMAS) @ basis_matrix(y_values).T


def sample_joint(size, seed):
    rng = np.random.default_rng(seed)
    upper_bound = 1 + 2 * SIGMAS.sum()
    x_parts = []
    y_parts = []
    n_accepted = 0

    while n_accepted < size:
        x = rng.uniform(-1, 1, size)
        y = rng.uniform(-1, 1, size)
        density_ratio = 1 + np.sum(
            basis_matrix(x) * SIGMAS * basis_matrix(y), axis=1
        )
        accepted = rng.uniform(size=size) < density_ratio / upper_bound
        x_parts.append(x[accepted])
        y_parts.append(y[accepted])
        n_accepted += accepted.sum()

    return np.concatenate(x_parts)[:size], np.concatenate(y_parts)[:size]


def scaled_factors(phi_model, psi_model, singular_values, x, y):
    scale = np.sqrt(np.maximum(singular_values, 0))
    return phi_model.predict(x[:, None]) * scale, psi_model.predict(y[:, None]) * scale


def subspace_error(estimated, exact):
    estimated = estimated - estimated.mean(axis=0)
    exact = exact - exact.mean(axis=0)
    estimated_basis = np.linalg.qr(estimated)[0][:, : exact.shape[1]]
    exact_basis = np.linalg.qr(exact)[0][:, : exact.shape[1]]
    difference = estimated_basis @ estimated_basis.T - exact_basis @ exact_basis.T
    return np.linalg.norm(difference, ord="fro") / np.sqrt(2 * exact.shape[1])


def orthogonality_error(values):
    centered = values - values.mean(axis=0)
    gram = centered.T @ centered / len(centered)
    return np.linalg.norm(gram - np.eye(gram.shape[0]), ord="fro") / np.sqrt(gram.shape[0])

We use

$$\kappa(x,y)=1+\sum_{j=1}^3\sigma_j e_j(x)e_j(y),\qquad e_j(t)=\sqrt{2}\cos(j\pi t),$$

with $(\sigma_1,\sigma_2,\sigma_3)=(0.18,0.16,0.12)$. Since $|e_j(x)e_j(y)|\leq 2$, the density ratio is at least $1-2\sum_j\sigma_j=0.08$.

In [2]:
SIGMAS = np.array([0.18, 0.16, 0.12])
RANK = len(SIGMAS)
N_TRAIN = 5_000
N_VALIDATION = 2_000
SEED = 0

x_train, y_train = sample_joint(N_TRAIN, seed=12)
x_validation, y_validation = sample_joint(N_VALIDATION, seed=99)

In [3]:
fsnm_grid = {
    "n_iterations": [10, 20, 40],
    "step_size": [0.05, 0.1, 0.2],
    "max_depth": [3, None],
    "min_samples_leaf": [100, 300],
}
fsnm_results = []
for parameters in ParameterGrid(fsnm_grid):
    candidate_phi, candidate_psi, candidate_values, _ = fit_fsnm(
        x_train, y_train, rank=RANK, seed=SEED, **parameters
    )
    phi_validation, psi_validation = scaled_factors(
        candidate_phi, candidate_psi, candidate_values, x_validation, y_validation
    )
    fsnm_results.append(
        {**parameters, "validation_loss": float(empirical_loss(phi_validation, psi_validation))}
    )
fsnm_results.sort(key=lambda result: result["validation_loss"])
fsnm_parameters = {name: fsnm_results[0][name] for name in fsnm_grid}

print(f"FSNM evaluated combinations: {len(fsnm_results)}")
print(f"FSNM best hyperparameters: {fsnm_parameters}")
print(f"FSNM validation loss: {fsnm_results[0]['validation_loss']:.4f}")

FSNM evaluated combinations: 36
FSNM best hyperparameters: {'n_iterations': 40, 'step_size': 0.1, 'max_depth': 3, 'min_samples_leaf': 300}
FSNM validation loss: -0.0516


In [4]:
phi_fsnm, psi_fsnm, values_fsnm, _ = fit_fsnm(
    x_train, y_train, rank=RANK, seed=SEED, **fsnm_parameters
)

grid = np.linspace(-1, 1, 160)
exact_basis = basis_matrix(grid)
kappa_true = kappa_exact(grid, grid)
phi_fsnm_grid = phi_fsnm.predict(grid[:, None])
psi_fsnm_grid = psi_fsnm.predict(grid[:, None])
kappa_fsnm = 1 + (phi_fsnm_grid * values_fsnm) @ psi_fsnm_grid.T

metrics = {
    "RMSE": np.sqrt(np.mean((kappa_fsnm - kappa_true) ** 2)),
    "spectrum error": np.linalg.norm(values_fsnm - SIGMAS),
    "subspace error": 0.5 * (
        subspace_error(phi_fsnm_grid, exact_basis)
        + subspace_error(psi_fsnm_grid, exact_basis)
    ),
    "orthogonality error": 0.5 * (
        orthogonality_error(phi_fsnm_grid)
        + orthogonality_error(psi_fsnm_grid)
    ),
}

print(f"True singular values: {SIGMAS}")
print(f"FSNM singular values: {np.round(values_fsnm, 4)}")
print("\nMetric                 FSNM")
for metric, value in metrics.items():
    print(f"{metric:20s} {value:9.4f}")

True singular values: [0.18 0.16 0.12]
FSNM singular values: [0.1824 0.1727 0.1083]

Metric                 FSNM
RMSE                    0.1807
spectrum error          0.0174
subspace error          0.5012
orthogonality error     0.0301


In [5]:
value_limits = (
    min(kappa_true.min(), kappa_fsnm.min()),
    max(kappa_true.max(), kappa_fsnm.max()),
)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
for axis, (values, title) in zip(
    axes[:2],
    [
        (kappa_true, r"True $\kappa$"),
        (kappa_fsnm, r"FSNM $\widehat\kappa$"),
    ],
):
    image = axis.imshow(
        values.T,
        origin="lower",
        extent=[-1, 1, -1, 1],
        cmap="coolwarm",
        vmin=value_limits[0],
        vmax=value_limits[1],
    )
    axis.set(title=title, xlabel="$x$", ylabel="$y$")
    fig.colorbar(image, ax=axis, shrink=0.82)

error = kappa_fsnm - kappa_true
error_limit = np.max(np.abs(error))
image = axes[2].imshow(
    error.T,
    origin="lower",
    extent=[-1, 1, -1, 1],
    cmap="coolwarm",
    vmin=-error_limit,
    vmax=error_limit,
)
axes[2].set(title=r"$\widehat\kappa-\kappa$", xlabel="$x$", ylabel="$y$")
fig.colorbar(image, ax=axes[2], shrink=0.82)

project_directory = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
figure_directory = project_directory / "figures"
figure_directory.mkdir(exist_ok=True)
figure_path = figure_directory / "00_rank3_close_spectrum.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
print(f"Figure saved to: {figure_path}")
fig

Figure saved to: /home/thiago/Projects/paper-fsnm/code/fsnm/figures/00_rank3_close_spectrum.png


<Figure size 1200x360 with 6 Axes>